# Ollama Cloud + Qwen 3.5 Demo

This notebook demonstrates using the OpenAI-compatible API from Ollama Cloud with the `qwen3.5:cloud` model.

**Run this notebook on JupyterHub** (deployed on the ML clusters) or locally after installing the `openai` package.

In [ ]:
%pip install -q openai

In [ ]:
import os
from openai import OpenAI

# Set your Ollama API key here
OLLAMA_API_KEY = os.environ.get("OLLAMA_API_KEY", "your-key-here")

if OLLAMA_API_KEY == "your-key-here":
    print("WARNING: Set your API key! Edit OLLAMA_API_KEY above or run:")
    print('  import os; os.environ["OLLAMA_API_KEY"] = "your-actual-key"')

client = OpenAI(
    base_url="https://ollama.com/v1",
    api_key=OLLAMA_API_KEY,
    timeout=60,
)

MODEL = "qwen3.5:cloud"
print(f"Client configured: {MODEL} @ {client.base_url}")

## Connectivity Test

Run this cell first to verify the cluster can reach Ollama Cloud.

In [ ]:
import urllib.request, urllib.error

req = urllib.request.Request(
    "https://ollama.com/v1/chat/completions",
    data=b'{"model":"qwen3.5:cloud","messages":[{"role":"user","content":"say ok"}],"stream":false}',
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {OLLAMA_API_KEY}",
    },
    method="POST",
)

try:
    with urllib.request.urlopen(req, timeout=30) as resp:
        print(f"Status: {resp.status}")
        print(resp.read().decode()[:500])
except urllib.error.HTTPError as e:
    print(f"HTTP Error: {e.code} {e.reason}")
    print(e.read().decode()[:200])
except Exception as e:
    print(f"Error: {e}")

## 1. Simple Chat Completion

In [ ]:
print("Sending request...")
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Do not use <think> tags. Be concise."},
        {"role": "user", "content": "Explain Kubernetes in 3 sentences."},
    ],
)

print(response.choices[0].message.content)

## 3. Tool Calling (Function Calling)

In [ ]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_cluster_status",
            "description": "Get the status of a Kubernetes cluster by name",
            "parameters": {
                "type": "object",
                "properties": {
                    "cluster_name": {
                        "type": "string",
                        "description": "The name of the cluster, e.g. team-ml-dev",
                    }
                },
                "required": ["cluster_name"],
            },
        },
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the status of the team-ml-dev cluster?"},
    ],
    tools=tools,
    tool_choice="auto",
)

msg = response.choices[0].message
if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f"Tool: {tc.function.name}")
        print(f"Args: {tc.function.arguments}")
else:
    print(msg.content)

## 4. Multi-turn Conversation

In [ ]:
messages = [
    {"role": "system", "content": "You are a Kubernetes expert. Be concise."},
    {"role": "user", "content": "What is a vcluster?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages)
assistant_msg = response.choices[0].message.content
print(f"Assistant: {assistant_msg}\n")

messages.append({"role": "assistant", "content": assistant_msg})
messages.append({"role": "user", "content": "How is it different from a namespace?"})

response = client.chat.completions.create(model=MODEL, messages=messages)
print(f"Assistant: {response.choices[0].message.content}")